# 🧪 Notebook Thử Nghiệm: Chuyển Đổi, Làm Sạch & Chuẩn Hóa Dữ Liệu (Bronze ➔ Silver)

Notebook này dùng để **khảo sát dữ liệu thực tế** và **thử nghiệm từng bước logic làm sạch & chuẩn hóa** trước khi đóng gói thành các module Python `.py` chính thức:

1. 📂 **Khám phá & Nạp dữ liệu Master Bronze**: Đọc 2 file `bronze_listings_master.json` và `bronze_details_valid.json`.
2. 🔗 **Thử nghiệm Ghép Bảng (Join Listings + Details)**: Kết hợp thông tin trang danh mục (Logo, tag ngoài) và trang chi tiết (JD sâu).
3. 💰 **Thử nghiệm Chuẩn hóa Mức Lương**: Tách `salary_min_vnd`, `salary_max_vnd`, xử lý tỷ giá USD, phát hiện lương thỏa thuận.
4. 🎓 **Thử nghiệm Chuẩn hóa Cấp Bậc (Seniority)**: Phân loại `Intern`, `Fresher`, `Junior`, `Middle`, `Senior`, `Lead`, `Manager`.
5. 📍 **Thử nghiệm Chuẩn hóa Địa Điểm**: Gom các chuỗi địa chỉ dài về `Hà Nội`, `Hồ Chí Minh`, `Đà Nẵng`, `Remote`, v.v.
6. 🔄 **Thử nghiệm Khử Trùng Chéo Sàn (Deduplication)**: Phát hiện tin đăng trùng giữa TopCV, TopDev, VietnamWorks.
7. 📊 **Preview Bảng Dữ Liệu Silver Hoàn Chỉnh**: Xem trước bảng phẳng Tabular sẵn sàng xuất sang Parquet.

In [1]:
import os
import re
import json
import pandas as pd
import numpy as np
from collections import Counter
from IPython.display import display, HTML

# Cấu hình hiển thị pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
print('Các thư viện đã sẵn sàng!')

Các thư viện đã sẵn sàng!


## 1. Nạp Dữ Liệu Từ Tầng Master Bronze

In [2]:
# Xác định đường dẫn file Bronze
possible_paths = [
    os.path.abspath(os.path.join("..", "data", "bronze")),
    os.path.abspath(os.path.join("data", "bronze"))
]
bronze_dir = next((p for p in possible_paths if os.path.exists(p)), None)

if not bronze_dir:
    raise FileNotFoundError("Không tìm thấy thư mục data/bronze!")

listings_path = os.path.join(bronze_dir, "bronze_listings_master.json")
details_path = os.path.join(bronze_dir, "bronze_details_valid.json")

print(f"📂 Đang đọc Listings từ: {listings_path}")
df_listings = pd.read_json(listings_path)

print(f"📂 Đang đọc Details từ : {details_path}")
df_details = pd.read_json(details_path)

print(f"\n📊 Tổng số tin Listings: {len(df_listings):,} dòng, {df_listings.shape[1]} cột")
print(f"📊 Tổng số tin Details : {len(df_details):,} dòng, {df_details.shape[1]} cột")

# Xem nhanh phân bố theo sàn
display(pd.DataFrame({
    'Số Listings': df_listings['source'].value_counts(),
    'Số Details Valid': df_details['source'].value_counts()
}))

📂 Đang đọc Listings từ: c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_listings_master.json
📂 Đang đọc Details từ : c:\Users\Minh Doan\repo_github\IT-Job-Market-Intelligence\data\bronze\bronze_details_valid.json

📊 Tổng số tin Listings: 1,746 dòng, 20 cột
📊 Tổng số tin Details : 1,698 dòng, 35 cột


,Số Listings,Số Details Valid
source,,
topcv,862,814
vietnamworks,764,764
topdev,120,120


## 2. Thử Nghiệm Ghép Bảng (Join Listings + Details)
Ghép theo cặp khóa `(source, source_job_id)` để giữ lại toàn bộ các trường bổ trợ của cả 2 bảng:

In [3]:
# Chọn các cột giá trị từ Listings mà Details có thể chưa có hoặc bổ sung
listing_cols = [
    'source', 'source_job_id', 'company_logo_url', 'skills_raw', 
    'posted_at_raw', 'raw_file_name'
]
df_listings_sub = df_listings[listing_cols].rename(columns={
    'skills_raw': 'listing_skills_raw',
    'raw_file_name': 'listing_file_name'
})

# Thực hiện Merge (Left join từ Details sang Listings)
df_merged = pd.merge(
    df_details,
    df_listings_sub,
    on=['source', 'source_job_id'],
    how='left'
)

match_rate = df_merged['listing_file_name'].notna().mean() * 100
print(f"✅ Tỷ lệ ghép nối thành công: {match_rate:.2f}% ({len(df_merged):,} dòng)")
display(df_merged[['source', 'source_job_id', 'title_raw', 'company_name_raw', 'company_logo_url', 'listing_file_name', 'raw_file_name']].head(3))

✅ Tỷ lệ ghép nối thành công: 100.00% (1,698 dòng)


,source,source_job_id,title_raw,company_name_raw,company_logo_url,listing_file_name,raw_file_name
0,topcv,2314643,Full-Stack Web Engineer Intern/Fresher (AI Coding: PHP / Ruby / Python),Công ty TNHH ZIGExN VeNtura,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/company_logos/67c94e...,page_001.json,TCV001.json
1,topcv,2314547,Kỹ Sư Hệ Thống Nhúng,Công ty TNHH Điện tử R&P,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/company_logos/cong-t...,page_001.json,TCV001.json
2,topcv,2321616,UI/UX Designer - Mobile App- [HCM] - Thu Nhập Lên Đến 30 Triệu,CÔNG TY TNHH FOXSCORE,https://cdn-new.topcv.vn/unsafe/https://static.topcv.vn/company_logos/ELdaFF...,page_001.json,TCV001.json


## 3. Khảo Sát & Thử Nghiệm Chuẩn Hóa Mức Lương (Salary Normalization)
Mục tiêu:
- Phát hiện trạng thái thương lượng (`is_negotiable`).
- Tách `salary_min_vnd` và `salary_max_vnd`.
- Tự động quy đổi USD sang VND (tỷ giá ước tính 25,400 VND/USD).
- Xử lý các dạng lương: triệu, VND số đầy đủ (`14000000`), tiền tố `Tới 30 triệu`, `Từ 15 triệu`.

In [4]:
EXCHANGE_RATE_USD_VND = 25400

def parse_salary_experimental(salary_str):
    if not salary_str or pd.isna(salary_str):
        return {'min_vnd': None, 'max_vnd': None, 'currency': 'UNKNOWN', 'is_negotiable': True}
    
    sal_lower = str(salary_str).lower().strip()
    
    # 1. Kiểm tra thương lượng / thỏa thuận
    if any(w in sal_lower for w in ['thỏa thuận', 'thoả thuận', 'thương lượng', 'negotiable', 'cạnh tranh']):
        return {'min_vnd': None, 'max_vnd': None, 'currency': 'NEGOTIABLE', 'is_negotiable': True}
    
    clean_str = sal_lower.replace(',', '')
    
    # Tìm dạng range: number (kèm tr/m tùy chọn) - number
    match_range = re.search(r'(-?\d+(?:\.\d+)?)\s*(?:tr|m|k)?\s*(?:-|đến|to)\s*(-?\d+(?:\.\d+)?)', clean_str)
    
    val_min = None
    val_max = None
    
    if match_range:
        val_min = float(match_range.group(1))
        val_max = float(match_range.group(2))
    else:
        # Tìm dạng tiền tố: 'Tới 30 triệu', 'Lên đến 50M'
        match_upto = re.search(r'(?:tới|lên đến|up to|tối đa|<)\s*(\d+(?:\.\d+)?)', clean_str)
        if match_upto:
            val_max = float(match_upto.group(1))
        else:
            # Tìm dạng tiền tố: 'Từ 15 triệu', 'Trên 20M'
            match_from = re.search(r'(?:từ|trên|from|>)\s*(\d+(?:\.\d+)?)', clean_str)
            if match_from:
                val_min = float(match_from.group(1))
            else:
                # Tìm số đơn lẻ (ví dụ: '15000000')
                match_single = re.search(r'(\d+(?:\.\d+)?)', clean_str)
                if match_single:
                    val_min = float(match_single.group(1))
                    val_max = val_min
    
    if val_min is None and val_max is None:
        return {'min_vnd': None, 'max_vnd': None, 'currency': 'UNKNOWN', 'is_negotiable': True}
    
    # Đảm bảo min <= max nếu có cả 2
    if val_min is not None and val_max is not None and val_min > val_max:
        val_min, val_max = val_max, val_min
        
    # 2. Nhận diện đơn vị tiền tệ & Quy đổi về VND
    check_val = val_max if val_max is not None else val_min
    
    has_trieu = any(u in sal_lower for u in ['triệu', 'tr']) or re.search(r'\d+\s*(?:m|tr)\b', sal_lower)
    has_usd = any(u in sal_lower for u in ['usd', 'đô']) or ('$' in sal_lower and not has_trieu)
    
    if has_usd:
        currency = 'USD'
        multiplier = EXCHANGE_RATE_USD_VND
    elif check_val >= 500_000:
        currency = 'VND'
        multiplier = 1  # Đã là số tiền VND đầy đủ
    elif has_trieu or check_val <= 300:
        currency = 'VND'
        multiplier = 1_000_000  # Đơn vị triệu đồng
    else:
        currency = 'VND'
        multiplier = 1
        
    min_vnd = int(val_min * multiplier) if val_min is not None else None
    max_vnd = int(val_max * multiplier) if val_max is not None else None
    
    return {
        'min_vnd': min_vnd,
        'max_vnd': max_vnd,
        'currency': currency,
        'is_negotiable': False
    }

# Thử nghiệm trên danh sách các mức lương thực tế đa dạng trong dataset
sample_salaries = [
    "Thương lượng", "Thoả thuận", "25 - 40 triệu", "Tới 30 triệu", "Từ 15 triệu",
    "15000000", "$ 1,202-2,600 /tháng", "$ 1,000-2,000 /tháng", "Up to $3,500",
    "14000000 - 18000000 MONTH", "$ 13tr-17tr /tháng", "20tr-40tr ₫/tháng", "30 - 20 triệu"
]

test_results = []
for s in sample_salaries:
    res = parse_salary_experimental(s)
    test_results.append({
        'Chuỗi Gốc (salary_raw)': s,
        'Lương Min (VND)': f"{res['min_vnd']:,}" if res['min_vnd'] else "-",
        'Lương Max (VND)': f"{res['max_vnd']:,}" if res['max_vnd'] else "-",
        'Loại Tiền': res['currency'],
        'Thương Lượng?': res['is_negotiable']
    })

display(pd.DataFrame(test_results))

,Chuỗi Gốc (salary_raw),Lương Min (VND),Lương Max (VND),Loại Tiền,Thương Lượng?
0,Thương lượng,-,-,NEGOTIABLE,True
1,Thoả thuận,-,-,NEGOTIABLE,True
2,25 - 40 triệu,"25,000,000","40,000,000",VND,False
3,Tới 30 triệu,-,"30,000,000",VND,False
4,Từ 15 triệu,"15,000,000",-,VND,False
5,15000000,"15,000,000","15,000,000",VND,False
6,"$ 1,202-2,600 /tháng","30,530,800","66,040,000",USD,False
7,"$ 1,000-2,000 /tháng","25,400,000","50,800,000",USD,False
8,"Up to $3,500","88,900,000","88,900,000",USD,False
9,14000000 - 18000000 MONTH,"14,000,000","18,000,000",VND,False


## 4. Khảo Sát & Thử Nghiệm Chuẩn Hóa Cấp Bậc (Seniority Extraction)
Bóc tách cấp bậc chuyên môn từ `title_raw` và `experience_raw` thành các nhóm chuẩn:
- `Intern`, `Fresher`, `Junior`, `Middle`, `Senior`, `Lead`, `Manager/Director`

In [5]:
def extract_seniority_experimental(title_str, exp_str=""):
    text = f"{str(title_str)} {str(exp_str)}".lower()
    
    # Thứ tự ưu tiên kiểm tra
    if re.search(r'\b(director|giám đốc|vp|head of)\b', text):
        return 'Director'
    if re.search(r'\b(manager|trưởng phòng|quản lý)\b', text):
        return 'Manager'
    if re.search(r'\b(lead|leader|trưởng nhóm|chuyên gia cao cấp|principal|architect)\b', text):
        return 'Lead'
    if re.search(r'\b(senior|sr\.?|chuyên viên cao cấp|senior developer)\b', text):
        return 'Senior'
    if re.search(r'\b(junior|jr\.?|chuyên viên sơ cấp)\b', text):
        return 'Junior'
    if re.search(r'\b(fresher|mới tốt nghiệp)\b', text):
        return 'Fresher'
    if re.search(r'\b(intern|internship|thực tập|thực tập sinh)\b', text):
        return 'Intern'
    if re.search(r'\b(middle|mid|chuyên viên|kỹ sư)\b', text):
        return 'Middle'
    
    return 'Not Specified'  # Chưa phân cấp rõ

# Áp dụng thử nghiệm trên toàn bộ tin
df_merged['seniority_level'] = df_merged.apply(
    lambda row: extract_seniority_experimental(row['title_raw'], row['experience_raw']),
    axis=1
)

print("📊 Phân bố cấp bậc trong bộ dữ liệu thực tế:")
display(df_merged['seniority_level'].value_counts().to_frame('Số Lượng Tin'))

📊 Phân bố cấp bậc trong bộ dữ liệu thực tế:


,Số Lượng Tin
seniority_level,
Not Specified,851
Middle,278
Manager,180
Senior,174
Lead,113
Intern,37
Director,25
Fresher,21
Junior,19


## 5. Khảo Sát & Thử Nghiệm Chuẩn Hóa Địa Điểm (Location Normalization)
Gom các chuỗi địa chỉ dài, chi tiết số nhà thành các khu vực trọng điểm:
- `Hà Nội`, `Hồ Chí Minh`, `Đà Nẵng`, `Cần Thơ`, `Hải Phòng`, `Remote`, `Khác`

In [6]:
def normalize_location_experimental(loc_str, work_mode_str=""):
    full_text = f"{str(loc_str)} {str(work_mode_str)}".lower()
    
    if 'remote' in full_text or 'từ xa' in full_text:
        return 'Remote'
    if any(k in full_text for k in ['hà nội', 'hanoi', 'ha noi', 'hn']):
        return 'Hà Nội'
    if any(k in full_text for k in ['hồ chí minh', 'ho chi minh', 'hcm', 'sài gòn', 'saigon', 'q.1', 'quận ']):
        return 'Hồ Chí Minh'
    if any(k in full_text for k in ['đà nẵng', 'da nang']):
        return 'Đà Nẵng'
    if any(k in full_text for k in ['hải phòng', 'hai phong']):
        return 'Hải Phòng'
    if any(k in full_text for k in ['cần thơ', 'can tho']):
        return 'Cần Thơ'
    if any(k in full_text for k in ['bình dương', 'binh duong']):
        return 'Bình Dương'
        
    return 'Khác / Chưa rõ'

df_merged['location_city'] = df_merged.apply(
    lambda row: normalize_location_experimental(row['location_raw'], row['work_mode_raw']),
    axis=1
)

print("📊 Phân bố địa điểm sau chuẩn hóa:")
display(df_merged['location_city'].value_counts().to_frame('Số Lượng Tin'))

📊 Phân bố địa điểm sau chuẩn hóa:


,Số Lượng Tin
location_city,
Hà Nội,1040
Hồ Chí Minh,493
Khác / Chưa rõ,114
Đà Nẵng,20
Hải Phòng,17
Remote,12
Bình Dương,2


## 6. Thử Nghiệm Khử Trùng Lặp Chéo Sàn (Cross-Portal Deduplication)
Tìm các tin tuyển dụng được đăng tải đồng thời trên nhiều hơn 1 sàn tuyển dụng dựa trên **dấu vân tay** (Fingerprint): `Normalized Title + Normalized Company`:

In [7]:
def build_job_fingerprint(title, company):
    t_clean = re.sub(r'[^\w\s]', '', str(title).lower()).strip()
    c_clean = re.sub(r'[^\w\s]', '', str(company).lower()).strip()
    for prefix in ['công ty cổ phần', 'công ty tnhh', 'tập đoàn', 'company limited', 'jsc', 'ltd']:
        c_clean = c_clean.replace(prefix, '').strip()
    return f"{c_clean}___{t_clean}"

df_merged['dedup_fingerprint'] = df_merged.apply(
    lambda row: build_job_fingerprint(row['title_raw'], row['company_name_raw']),
    axis=1
)

# Tìm các fingerprint xuất hiện ở nhiều nguồn khác nhau
cross_dups = df_merged.groupby('dedup_fingerprint')['source'].nunique()
multi_source_fingerprints = cross_dups[cross_dups > 1].index

print(f"🔍 Số lượng tin trùng lặp xuất hiện trên từ 2 sàn trở lên: {len(multi_source_fingerprints):,} nhóm tin")

if len(multi_source_fingerprints) > 0:
    sample_fp = multi_source_fingerprints[0]
    sample_dups = df_merged[df_merged['dedup_fingerprint'] == sample_fp]
    print("\nVí dụ 1 tin trùng chéo sàn được phát hiện:")
    display(sample_dups[['source', 'title_raw', 'company_name_raw', 'salary_raw', 'location_city']])

🔍 Số lượng tin trùng lặp xuất hiện trên từ 2 sàn trở lên: 12 nhóm tin

Ví dụ 1 tin trùng chéo sàn được phát hiện:


,source,title_raw,company_name_raw,salary_raw,location_city
778,topdev,System Security Engineer (Senior),"CyberLogitec Vietnam Co., Ltd.",Thương lượng,Remote
874,vietnamworks,System Security Engineer (Senior),"CyberLogitec Vietnam Co., Ltd.",Thương lượng,Hồ Chí Minh


## 7. Preview Bảng Dữ Liệu Silver Hoàn Chỉnh
Tổng hợp tất cả các trường đã chuẩn hóa vào DataFrame Silver hoàn chỉnh trong bộ nhớ (hoàn toàn không lưu file ra ổ đĩa) để bạn quan sát, phân tích và đánh giá:

In [8]:
# Áp dụng toàn bộ chuẩn hóa lương vào DataFrame
salary_parsed_list = df_merged['salary_raw'].apply(parse_salary_experimental)
df_silver = df_merged.copy()

df_silver['salary_min_vnd'] = [s['min_vnd'] for s in salary_parsed_list]
df_silver['salary_max_vnd'] = [s['max_vnd'] for s in salary_parsed_list]
df_silver['salary_currency'] = [s['currency'] for s in salary_parsed_list]
df_silver['is_negotiable'] = [s['is_negotiable'] for s in salary_parsed_list]

# Lựa chọn các cột đại diện cho bảng Silver sạch
silver_display_cols = [
    'source', 'title_raw', 'seniority_level', 'company_name_raw',
    'location_city', 'salary_raw', 'salary_min_vnd', 'salary_max_vnd',
    'is_negotiable', 'listing_file_name', 'raw_file_name'
]

print(f"✨ Đã xây dựng thành công bảng Silver mẫu ({len(df_silver):,} dòng trong bộ nhớ - KHÔNG XUẤT FILE)!")
display(df_silver[silver_display_cols].sample(10, random_state=42))

# Thống kê nhanh phân bố lương của các tin có số cụ thể
sal_stats = df_silver[~df_silver['is_negotiable']][['salary_min_vnd', 'salary_max_vnd']].describe()
print("\n📊 Thống kê mức lương (VND) của các tin có khoảng lương cụ thể:")
display(sal_stats.map(lambda x: f'{x:,.0f}' if pd.notna(x) else '-'))

✨ Đã xây dựng thành công bảng Silver mẫu (1,698 dòng trong bộ nhớ - KHÔNG XUẤT FILE)!


,source,title_raw,seniority_level,company_name_raw,location_city,salary_raw,salary_min_vnd,salary_max_vnd,is_negotiable,listing_file_name,raw_file_name
618,topcv,"Manual Software Testing (Saas,Product E-Commerce)",Not Specified,CÔNG TY CỔ PHẦN CÔNG NGHỆ UPBASE,Hà Nội,13000000,13000000.0,13000000.0,False,page_009.json,TCV010.json
115,topcv,Nhân Viên Kinh Doanh Thị Trường / Triển Khai Phần Mềm - Từ 1 Năm Kinh Nghiệm...,Not Specified,CÔNG TY TNHH SENTO,Khác / Chưa rõ,20000000,20000000.0,20000000.0,False,page_004.json,TCV004.json
135,topcv,IT Sales - Tiếng Nhật - Open For Fresher [HCM - HN],Fresher,CÔNG TY TNHH SYSTEMEXE VIỆT NAM,Hồ Chí Minh,Thoả thuận,NaN,NaN,True,page_004.json,TCV004.json
350,topcv,Lập Trình Viên .NET ( Full Level),Not Specified,Công ty Cổ phần Atomi Digital,Hà Nội,Thoả thuận,NaN,NaN,True,page_007.json,TCV007.json
1395,vietnamworks,Business Analyst (BA),Not Specified,Công ty cổ phần công nghệ số VCI,Hà Nội,20tr-40tr ₫/tháng,20000000.0,40000000.0,False,page_007.json,VNW007.json
1447,vietnamworks,Giám Đốc Chi Nhánh Hồ Chí Minh (An Ninh Quốc Phòng/ Chính Phủ/ CNTT/ Thiết B...,Director,Công ty cổ phần đầu tư và công nghệ HTI,Hồ Chí Minh,Thương lượng,NaN,NaN,True,page_008.json,VNW008.json
669,topcv,Thực Tập Sinh Thiết Kế (Design Intern),Intern,Công ty Cổ phần Navee,Hồ Chí Minh,Tới 0.5 triệu,NaN,500000.0,False,page_010.json,TCV010.json
1686,vietnamworks,Business Analyst - Sales & CRM,Not Specified,Japfa Comfeed Vietnam Ltd,Bình Dương,Thương lượng,NaN,NaN,True,page_001.json,VNW001.json
1448,vietnamworks,IT Security Manager,Manager,Công Ty Cổ Phần Dược Phẩm Pharmacity,Hồ Chí Minh,Thương lượng,NaN,NaN,True,page_008.json,VNW008.json
772,topdev,Chuyên Gia Vận Hành Thẻ,Not Specified,AEON Finance,Hà Nội,Thương lượng,NaN,NaN,True,page_001.json,TC001.json



📊 Thống kê mức lương (VND) của các tin có khoảng lương cụ thể:


,salary_min_vnd,salary_max_vnd
count,591,689
mean,"24,251,976","32,255,230"
std,"42,848,840","58,957,720"
min,400,400
25%,"10,000,000","12,000,000"
50%,"15,000,000","20,000,000"
75%,"25,000,000","35,000,000"
max,"609,600,000","736,600,000"


## 8. Mô Hình Hóa & Phân Tách 2 Bảng Cốt Lõi: `companies` và `jobs`

Theo mô hình **Silver Layer (Normalized Relational Model)**:
1. 🏢 **Bảng `companies` (Dimension)**: Phân tách thực thể công ty thành bảng danh mục riêng biệt. Khử trùng các biến thể tên viết hoa/thường, loại bỏ tiền tố pháp lý (*Công ty Cổ phần, TNHH...*), bóc tách quy mô nhân viên (*min/max*), ngành nghề, và cấp phát `company_id` duy nhất (`UUIDv5` cố định).
2. 💼 **Bảng `jobs` (Fact / Core)**: Bảng thực thể trung tâm chứa đầy đủ thông tin chuẩn hóa của tin tuyển dụng (chức danh, cấp bậc, lương `min/max/currency/type`, kinh nghiệm `min/max năm`, trình độ học vấn, hình thức làm việc `work_mode`), và liên kết khóa ngoại `company_id` sang bảng `companies`.

In [9]:
# --- 8.1. Xây Dựng Bảng Dimension: companies ---
import uuid

def clean_company_name(name):
    if not name or pd.isna(name):
        return "Unknown Company"
    name = str(name).strip()
    prefixes = [
        r"^CÔNG TY TNHH MTV\s+", r"^CÔNG TY TNHH\s+", r"^Công ty TNHH MTV\s+", r"^Công ty TNHH\s+",
        r"^CÔNG TY CỔ PHẦN\s+", r"^Công ty Cổ phần\s+", r"^Công Ty Cổ Phần\s+", r"^Công Ty TNHH\s+",
        r"^Tập đoàn\s+", r"^TẬP ĐOÀN\s+", r"^Chi nhánh\s+",
    ]
    for p in prefixes:
        name = re.sub(p, "", name, flags=re.IGNORECASE)
    suffixes = [
        r"\s+-\s+Chi nhánh.*$", r"\s+-\s+CN.*$", r"\s+JSC$", r"\s+Co\.,\s*Ltd\.?$",
    ]
    for s in suffixes:
        name = re.sub(s, "", name, flags=re.IGNORECASE)
    return name.strip()

def parse_company_size(size_str):
    if not size_str or pd.isna(size_str):
        return None, None
    s = str(size_str).strip()
    range_match = re.search(r"(\d+)\s*[-–]\s*(\d+)", s)
    if range_match:
        return int(range_match.group(1)), int(range_match.group(2))
    plus_match = re.search(r"(\d+)\s*\+", s)
    if plus_match:
        return int(plus_match.group(1)), None
    more_match = re.search(r"(?:hơn|trên)\s*(\d+)", s, re.IGNORECASE)
    if more_match:
        return int(more_match.group(1)), None
    less_match = re.search(r"(?:ít hơn|dưới|<\s*)(\d+)", s, re.IGNORECASE)
    if less_match:
        return 1, int(less_match.group(1))
    return None, None

# Trích xuất và khử trùng thực thể công ty
company_records = []
company_key_to_id = {}

for idx, row in df_merged.iterrows():
    raw_name = str(row.get('company_name_raw', '')).strip()
    norm_name = clean_company_name(raw_name)
    if not norm_name or norm_name == "Unknown Company":
        continue
    
    comp_key = norm_name.lower()
    if comp_key not in company_key_to_id:
        # Sinh UUIDv5 deterministic theo tên công ty chuẩn hóa
        comp_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"company:{comp_key}"))[:18]
        company_key_to_id[comp_key] = comp_id
        
        comp_url = str(row.get('company_url', '') or '').strip()
        comp_logo = str(row.get('company_logo_url', '') or '').strip()
        comp_size_raw = str(row.get('company_size_raw', '') or '').strip()
        comp_loc_raw = str(row.get('company_location_raw', '') or '').strip()
        industry_raw = str(row.get('industry_raw', '') or '').strip()
        
        size_min, size_max = parse_company_size(comp_size_raw)
        
        company_records.append({
            'company_id': comp_id,
            'company_name_raw': raw_name,
            'company_name_normalized': norm_name,
            'company_url': comp_url,
            'company_logo_url': comp_logo,
            'company_size_raw': comp_size_raw,
            'company_size_min': size_min,
            'company_size_max': size_max,
            'company_location_raw': comp_loc_raw,
            'company_city': normalize_location_experimental(comp_loc_raw),
            'industry_raw': industry_raw,
            'industry_normalized': industry_raw if industry_raw else "Công nghệ thông tin / IT"
        })

df_companies = pd.DataFrame(company_records).drop_duplicates(subset=['company_id'])
print(f"🏢 Bảng companies: Đã gom nhóm và chuẩn hóa thành công {len(df_companies):,} công ty độc nhất từ {len(df_merged):,} tin!")
display(df_companies[['company_id', 'company_name_raw', 'company_name_normalized', 'company_size_min', 'company_size_max', 'company_city', 'industry_normalized']].head(5))


🏢 Bảng companies: Đã gom nhóm và chuẩn hóa thành công 890 công ty độc nhất từ 1,698 tin!


,company_id,company_name_raw,company_name_normalized,company_size_min,company_size_max,company_city,industry_normalized
0,b361832f-901f-5e4c,Công ty TNHH ZIGExN VeNtura,ZIGExN VeNtura,100.0,499.0,Khác / Chưa rõ,"IT phần mềm, Công nghệ thông tin"
1,607e1460-1cd0-59f6,Công ty TNHH Điện tử R&P,Điện tử R&P,25.0,99.0,Hồ Chí Minh,"IT phần mềm, Công nghệ thông tin, Điện tử viễn thông"
2,b155e69a-30db-5d6b,CÔNG TY TNHH FOXSCORE,FOXSCORE,25.0,99.0,Hồ Chí Minh,"IT phần mềm, Công nghệ thông tin"
3,20f3cd62-0d47-582a,CÔNG TY CỔ PHẦN THƯƠNG MẠI QUỐC TẾ LB GLOBAL,THƯƠNG MẠI QUỐC TẾ LB GLOBAL,25.0,99.0,Hà Nội,"Dịch vụ khách hàng, Tư vấn, Kinh doanh / Bán hàng"
4,10d16130-e0cf-5de2,CÔNG TY TNHH VODAPLAY VIỆT NAM,VODAPLAY VIỆT NAM,25.0,99.0,Hà Nội,"IT phần mềm, Công nghệ thông tin"


In [10]:
# --- 8.2. Xây Dựng Bảng Core: jobs ---

def clean_job_title(title):
    if not title or pd.isna(title):
        return ""
    t = str(title).strip()
    t = re.sub(r'\[.*?\]', '', t)  # Bỏ [HCM], [Hà Nội], v.v.
    # Bỏ các cụm từ quảng cáo thu nhập / đãi ngộ ở đuôi
    noise_patterns = [
        r'(?:-|\|)?\s*(?:thu nhập|lương|up to|upto|lên đến|tới)\s*.*$',
        r'(?:-|\|)?\s*(?:chế độ tốt|hấp dẫn|hot job|urgent|thưởng).*$',
    ]
    for np in noise_patterns:
        t = re.sub(np, '', t, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', t).strip(' -|/,')

def parse_experience_years(exp_str):
    if not exp_str or pd.isna(exp_str):
        return None, None
    s = str(exp_str).strip().lower()
    if any(k in s for k in ['không yêu cầu', 'chưa có kinh nghiệm', 'không cần']):
        return 0.0, 0.0
    m_range = re.search(r"(\d+(?:\.\d+)?)\s*[-–]\s*(\d+(?:\.\d+)?)\s*năm", s)
    if m_range:
        return float(m_range.group(1)), float(m_range.group(2))
    m_min = re.search(r"(?:trên|từ|tối thiểu|\+)\s*(\d+(?:\.\d+)?)\s*năm|(\d+(?:\.\d+)?)\s*(?:\+|năm trở lên)", s)
    if m_min:
        val = m_min.group(1) or m_min.group(2)
        return float(val), None
    m_max = re.search(r"dưới\s*(\d+(?:\.\d+)?)\s*năm", s)
    if m_max:
        return 0.0, float(m_max.group(1))
    m_exact = re.search(r"(\d+(?:\.\d+)?)\s*năm", s)
    if m_exact:
        return float(m_exact.group(1)), float(m_exact.group(1))
    return None, None

def normalize_education(edu_str):
    if not edu_str or pd.isna(edu_str):
        return "Không yêu cầu"
    s = str(edu_str).strip().lower()
    if 'đại học' in s or 'cử nhân' in s or 'bachelor' in s:
        return "Đại học"
    if 'cao đẳng' in s or 'college' in s:
        return "Cao đẳng"
    if 'thạc sĩ' in s or 'master' in s:
        return "Thạc sĩ"
    if 'tiến sĩ' in s or 'phd' in s:
        return "Tiến sĩ"
    if 'trung cấp' in s or 'chứng chỉ' in s:
        return "Trung cấp / Chứng chỉ"
    return "Không yêu cầu"

def normalize_employment_type(emp_str):
    if not emp_str or pd.isna(emp_str):
        return "Full-time"
    s = str(emp_str).strip().lower()
    if any(k in s for k in ['toàn thời gian', 'full-time', 'full time']):
        return "Full-time"
    if any(k in s for k in ['bán thời gian', 'part-time', 'part time']):
        return "Part-time"
    if any(k in s for k in ['thực tập', 'internship', 'intern']):
        return "Internship"
    if any(k in s for k in ['hợp đồng', 'contract', 'freelance']):
        return "Contract"
    return "Full-time"

job_records = []

for idx, row in df_merged.iterrows():
    source = row.get('source', '')
    source_job_id = row.get('source_job_id', '')
    raw_name = str(row.get('company_name_raw', '')).strip()
    norm_name = clean_company_name(raw_name)
    comp_id = company_key_to_id.get(norm_name.lower(), None)
    
    # Deterministic UUIDv5 cho job_id
    job_uuid = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{source}:{source_job_id}"))[:18]
    
    # Chuẩn hóa mức lương & loại lương
    sal_parsed = parse_salary_experimental(row.get('salary_raw', ''))
    s_min = sal_parsed['min_vnd']
    s_max = sal_parsed['max_vnd']
    s_curr = "VND"
    s_period = "month"
    raw_sal = str(row.get('salary_raw', '')).lower()
    if sal_parsed['is_negotiable'] or any(k in raw_sal for k in ['thỏa thuận', 'thoả thuận', 'cạnh tranh', 'negotiable']):
        s_type = "negotiable"
    elif s_min is not None and s_max is not None:
        s_type = "fixed" if s_min == s_max else "range"
    elif s_max is not None:
        s_type = "max_only"
    elif s_min is not None:
        s_type = "min_only"
    else:
        s_type = "undisclosed"
    
    # Kinh nghiệm, học vấn, cấp bậc, hình thức
    e_min, e_max = parse_experience_years(row.get('experience_raw', ''))
    edu_level = normalize_education(row.get('education_raw', ''))
    seniority = extract_seniority_experimental(row.get('title_raw', ''), row.get('experience_raw', ''))
    city = normalize_location_experimental(row.get('location_raw', ''), row.get('work_mode_raw', ''))
    emp_type = normalize_employment_type(row.get('employment_type_raw', ''))
    
    wm_raw = (str(row.get('work_mode_raw', '')) + ' ' + str(row.get('location_raw', ''))).lower()
    work_mode = "Remote" if any(k in wm_raw for k in ['remote', 'từ xa']) else ("Hybrid" if 'hybrid' in wm_raw else "Onsite")
    
    job_records.append({
        'job_id': job_uuid,
        'source': source,
        'source_job_id': source_job_id,
        'job_url': row.get('job_url', ''),
        'title_raw': row.get('title_raw', ''),
        'title_normalized': clean_job_title(row.get('title_raw', '')),
        'seniority': seniority,
        'company_id': comp_id,
        'location_raw': row.get('location_raw', ''),
        'city': city,
        'salary_raw': row.get('salary_raw', ''),
        'salary_min': s_min,
        'salary_max': s_max,
        'salary_currency': s_curr,
        'salary_period': s_period,
        'salary_type': s_type,
        'experience_raw': row.get('experience_raw', ''),
        'experience_min_years': e_min,
        'experience_max_years': e_max,
        'education_raw': row.get('education_raw', ''),
        'education_level': edu_level,
        'work_mode_raw': row.get('work_mode_raw', ''),
        'work_mode': work_mode,
        'employment_type': emp_type,
        'working_time_raw': row.get('working_time_raw', ''),
        'published_at': row.get('published_at_raw', ''),
        'updated_at': row.get('updated_at_raw', ''),
        'deadline': row.get('deadline_raw', ''),
        'first_seen_at': row.get('crawled_at', ''),
        'last_seen_at': row.get('crawled_at', ''),
        'is_active': True,
        'data_source_id': source,
        'dq_status': row.get('dq_status', 'valid')
    })

df_jobs = pd.DataFrame(job_records)
print(f"💼 Bảng jobs: Đã tạo {len(df_jobs):,} bản ghi việc làm chuẩn hóa!")

# Kiểm tra tính toàn vẹn khóa ngoại (FK Integrity)
valid_fks = df_jobs['company_id'].isin(df_companies['company_id']).sum()
print(f"🔗 Kiểm tra toàn vẹn khóa ngoại (FK Integrity): {valid_fks}/{len(df_jobs)} ({valid_fks/len(df_jobs)*100:.1f}%) tin có company_id khớp chuẩn với bảng companies!")

print("\n🔍 So sánh Title Gốc vs Title Chuẩn Hóa:")
display(df_jobs[['title_raw', 'title_normalized', 'seniority']].sample(5, random_state=42))

print("\n🔍 Preview 5 dòng dữ liệu của bảng jobs:")
display(df_jobs[['job_id', 'title_normalized', 'seniority', 'company_id', 'city', 'salary_min', 'salary_max', 'salary_type', 'experience_min_years', 'education_level', 'work_mode']].head(5))

print("\n📊 Phân bố loại lương (salary_type):")
display(df_jobs['salary_type'].value_counts())

print("\n📊 Phân bố cấp bậc (seniority):")
display(df_jobs['seniority'].value_counts())

print("\n📊 Phân bố trình độ học vấn (education_level):")
display(df_jobs['education_level'].value_counts())


💼 Bảng jobs: Đã tạo 1,698 bản ghi việc làm chuẩn hóa!
🔗 Kiểm tra toàn vẹn khóa ngoại (FK Integrity): 1698/1698 (100.0%) tin có company_id khớp chuẩn với bảng companies!

🔍 So sánh Title Gốc vs Title Chuẩn Hóa:


,title_raw,title_normalized,seniority
618,"Manual Software Testing (Saas,Product E-Commerce)","Manual Software Testing (Saas,Product E-Commerce)",Not Specified
115,Nhân Viên Kinh Doanh Thị Trường / Triển Khai Phần Mềm - Từ 1 Năm Kinh Nghiệm...,Nhân Viên Kinh Doanh Thị Trường / Triển Khai Phần Mềm - Từ 1 Năm Kinh Nghiệm,Not Specified
135,IT Sales - Tiếng Nhật - Open For Fresher [HCM - HN],IT Sales - Tiếng Nhật - Open For Fresher,Fresher
350,Lập Trình Viên .NET ( Full Level),Lập Trình Viên .NET ( Full Level),Not Specified
1395,Business Analyst (BA),Business Analyst (BA),Not Specified



🔍 Preview 5 dòng dữ liệu của bảng jobs:


,job_id,title_normalized,seniority,company_id,city,salary_min,salary_max,salary_type,experience_min_years,education_level,work_mode
0,632db9af-f938-516f,Full-Stack Web Engineer Intern/Fresher (AI Coding: PHP / Ruby / Python),Fresher,b361832f-901f-5e4c,Hồ Chí Minh,NaN,NaN,negotiable,0.0,Đại học,Onsite
1,59053e61-8651-55d8,Kỹ Sư Hệ Thống Nhúng,Middle,607e1460-1cd0-59f6,Hồ Chí Minh,NaN,NaN,negotiable,0.0,Cao đẳng,Onsite
2,1d1cefc2-ca21-59db,UI/UX Designer - Mobile App,Not Specified,b155e69a-30db-5d6b,Hồ Chí Minh,15000000.0,15000000.0,fixed,2.0,Đại học,Onsite
3,b809a648-23a1-50c6,Business Analyst - Data Hệ Thống Phân Phối,Not Specified,20f3cd62-0d47-582a,Hà Nội,12000000.0,12000000.0,fixed,NaN,Cao đẳng,Onsite
4,f6630770-bb45-5db2,Junior QA Tester (Manual Testing),Junior,10d16130-e0cf-5de2,Hà Nội,NaN,16000000.0,max_only,1.0,Cao đẳng,Onsite



📊 Phân bố loại lương (salary_type):


salary_type
negotiable    1002
fixed          416
range          168
max_only       105
min_only         7
Name: count, dtype: int64


📊 Phân bố cấp bậc (seniority):


seniority
Not Specified    851
Middle           278
Manager          180
Senior           174
Lead             113
Intern            37
Director          25
Fresher           21
Junior            19
Name: count, dtype: int64


📊 Phân bố trình độ học vấn (education_level):


education_level
Đại học                  1008
Không yêu cầu             403
Cao đẳng                  238
Trung cấp / Chứng chỉ      45
Thạc sĩ                     3
Tiến sĩ                     1
Name: count, dtype: int64